In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer #导入TF-IDF向量化工具
from sklearn.model_selection import train_test_split #导入数据集划分工具
from sklearn.metrics import classification_report #导入分类报告工具
from scipy.sparse import hstack #导入稀疏矩阵水平拼接工具

In [9]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [2]:
import pandas as pd

# 读取训练集和测试集数据
train=pd.read_csv('data/train.csv')
test=pd.read_csv('data/test.csv')
submission=pd.read_csv('data/sample_submission.csv')


In [3]:
print(train.isnull().sum())   #检查训练集中的缺失值，.sum()函数用于统计每一列中缺失值的数量
print(test.isnull().sum())     
print(test.duplicated().sum())
print(train.duplicated().sum())


total_id = len(train["id"]) #计算训练数据中'id'列的总行数
total_unique_id = len(train["id"].unique()) #计算训练数据中'id'列的唯一值数量

print("Total number of 'id' duplicates:",total_id - total_unique_id) #输出训练数据中'id'列的重复值数量


id                0
model_a           0
model_b           0
prompt            0
response_a        0
response_b        0
winner_model_a    0
winner_model_b    0
winner_tie        0
dtype: int64
id            0
prompt        0
response_a    0
response_b    0
dtype: int64
0
0
Total number of 'id' duplicates: 0


数据预处理

In [4]:
def which_winner(value):
    if  value["winner_model_a"] == 1:
         #winner model a
         value["winner_model_b"] = 0
         value["winner_tie"] = 0
         return 0
    elif value["winner_model_b"] == 1:
         #winner model b
         return 1
    elif value["winner_tie"] == 1:
         #winner tie
         return 2
    return None

train["winner"] = train.apply(which_winner, axis=1)

train["winner_model"] = train["winner"].astype(str) #将'winner'列转换为字符串类型，并赋值给'winner_model'列
#将'winner_model'列中的值进行替换，将"0"替换为"model a"，将"1"替换为"model b"，将"2"替换为"winner tie"
#loc方法用于根据条件选择行，并对选定的行进行赋值操作
train.loc[train["winner_model"] == "0", "winner_model"] = "model a"
train.loc[train["winner_model"] == "1", "winner_model"] = "model b"
train.loc[train["winner_model"] == "2", "winner_model"] = "winner tie"
# value_counts()方法用于统计'winner_model'列中每个唯一值的出现次数，并返回一个包含唯一值及其对应计数的Series对象
result_model_winner = train["winner_model"].value_counts()

BERT

In [5]:
from transformers import BertTokenizer, BertModel #导入BERT分词器和BERT模型
import torch #导入PyTorch库
from torch.utils.data import Dataset, DataLoader #导入PyTorch的数据集和数据加载器工具

# 定义数据集类
class TextDataset(Dataset):
    def __init__(self,texts):
        self.texts=texts

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self,idx):
        return self.texts[idx]

d:\练习\python\Machine Learning\LLM Classification Finetuning\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# 定义函数以批处理方式提取BERT特征
def extract_bert_embeddings(texts,tokenizer,model,batch_size=16,max_length=128,device='cuda'):
    """
    输入：
    - texts: 文本列表
    - tokenizer: BERT分词器
    - model: BERT模型
    - batch_size: 批处理大小
    - max_length: 文本最大长度
    - device: 计算设备（'cuda'或'cpu'）
    输出：
    - features: BERT特征矩阵
    """
    dataset=TextDataset(texts)
    # DataLoader是PyTorch中用于加载数据的工具，可以将数据集分成小批量，并在训练过程中进行迭代
    # 参数说明：
    # - dataset: 数据集
    # - batch_size: 每个批次的样本数量
    # - shuffle: 是否在每个epoch开始时打乱数据
    dataloader=DataLoader(dataset,batch_size=batch_size,shuffle=False)
    
    all_embeddings=[] #用于存储所有文本的BERT特征
    with torch.no_grad(): #在提取特征时不计算梯度，以节省内存和计算资源
        for batch in dataloader:
            # 将文本批次转换为BERT输入格式，包括分词、添加特殊标记、填充和截断
            encoded=tokenizer(batch,padding=True,truncation=True,max_length=max_length,return_tensors='pt').to(device)
            outputs=model(**encoded) #将编码后的输入传递给BERT模型，获取输出
            cls_embeddings=outputs.last_hidden_state[:,0,:] #提取每个文本的CLS向量
            all_embeddings.append(cls_embeddings.cpu()) #将CLS向量移回CPU并存储
    return torch.cat(all_embeddings,dim=0) #将所有批次的CLS向量拼接成一个矩阵并返回

In [ ]:
# 加载分词器
tokenizer=BertTokenizer.from_pretrained('distilbert-base-uncased') #加载DistilBERT分词器
# 加载模型  
bert_model=BertModel.from_pretrained('distilbert-base-uncased').to('cuda') #加载DistilBERT模型并移动到GPU


d:\练习\python\Machine Learning\LLM Classification Finetuning\env\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\XP\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You are using a model of type `distilbert` to instantiate a model of type `bert`. Thi

In [ ]:

#将'prompt'列和'response_a'列的文本进行拼接，并赋值给新的列'prompt_response_a'
train['prompt_response_a']=train['prompt']+'[SEP]'+train['response_a'] 
train['prompt_response_b']=train['prompt']+'[SEP]'+train['response_b']

# 通过优化批处理提取特征
batch_size=16
embeddings_a=extract_bert_embeddings(
    list(train['prompt_response_a']),tokenizer,bert_model,batch_size=batch_size,device='cuda'
)
embeddings_b=extract_bert_embeddings(
    list(train['prompt_response_b']),tokenizer,bert_model,batch_size=batch_size,device='cuda'
)
    
print(embeddings_a.shape) #输出提取的BERT特征矩阵的形状
print(embeddings_b.shape)

KeyboardInterrupt: 

In [ ]:
import torch.nn.functional as F
combined_features_concat=torch.cat((embeddings_a,embeddings_b),dim=1) #将两个特征矩阵在列方向上拼接，形成一个新的特征矩阵
combined_features_diff=embeddings_a - embeddings_b #计算两个特征矩阵的差异，形成一个新的特征矩阵
cosine_sim=F.cosine_similarity(embeddings_a,embeddings_b,dim=1) #计算两个特征矩阵之间的余弦相似度，返回一个包含相似度值的张量


In [ ]:
train_y=train['winner'] #提取训练数据中的标签列'winner'，并赋值给train_y变量
train_X=combined_features_concat #将拼接后的特征矩阵赋值给train_X变量，作为训练数据的特征输入

In [ ]:
from sklearn.linear_model import LogisticRegression
from datetime import datetime
from sklearn.metrics import confusion_matrix, classification_report

#start time to calculate the execution time
start_time = datetime.now()

print("Use logistic regression")
#Apply the Logistic Regression
model = LogisticRegression(max_iter=500, multi_class='multinomial', solver='saga')
model.fit(train_X, train_y)
#end time
end_time = datetime.now()
#calculate the execution time
execution_time = (end_time - start_time).total_seconds()
print(f"The execution time is : {execution_time} secondes")


# Split into validation and training data
train_X_train, train_X_val, train_y_train, train_y_val = train_test_split(
    train_X, train_y, test_size=0.2, random_state=42
)

# Record start time to calculate the execution time
start = datetime.now()

# Make predictions on the validation set
value_y_predict = model.predict(train_X_val)
print('Model predicted values:', value_y_predict)
print('True values:', train_y_val)

# Predicted probabilities
value_y_probabilities = model.predict_proba(train_X_val)
print('Model prediction probabilities (class-wise):\n', value_y_probabilities)

# Model accuracy
score = model.score(train_X_val, train_y_val)
print('The Model Accuracy Score:', score)

# Confusion matrix
conf_matrix = confusion_matrix(train_y_val, value_y_predict)
print("Confusion Matrix:\n", conf_matrix)

# Classification report
report = classification_report(train_y_val, value_y_predict)  # Arguments fixed
print("Classification Report:\n", report)
